# Notebook 01 — Data Ingestion & Preprocessing

**Goal**: Load EuroVerdict, PolyTruth, and Hugging Face (`rares127/dezinformare-ro`) datasets,
clean and standardise them, then create stratified 80/10/10 train/val/test splits.

**Output schema**: `claim_id`, `claim_text`, `evidence_text`, `veracity_label`, `justification`

Splits saved to `data/processed/`.

In [ ]:
# Install dependencies (run once)
# %pip install datasets pandas scikit-learn beautifulsoup4 requests tqdm

In [ ]:
import re
import uuid
import unicodedata
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
# Pre-create all output directories so later notebooks never hit a missing-dir error
for _d in [PROCESSED_DIR, RAW_DIR,
           Path('../data/models'),
           Path('../data/figures')]:
    _d.mkdir(parents=True, exist_ok=True)

LABEL_MAP_NORM = {
    # EuroVerdict labels
    'true': 'true',
    'adevarat': 'true',
    'adevărat': 'true',
    'false': 'false',
    'fals': 'false',
    'partially true': 'partially_true',
    'partial true': 'partially_true',
    'parțial adevărat': 'partially_true',
    'partial adevarat': 'partially_true',
    'misleading': 'partially_true',
    'înşelător': 'partially_true',
    # PolyTruth / dezinformare-ro labels
    '0': 'false',
    '1': 'true',
    '2': 'partially_true',
}

print('Paths ready.')

## 1. Cleaning Utilities

In [ ]:
def remove_html(text: str) -> str:
    """Strip HTML tags using a simple regex (no external dep needed)."""
    return re.sub(r'<[^>]+>', ' ', str(text)).strip()


# Romanian diacritic standardisation:
# Normalise ş→ș, ţ→ț (cedilla → comma-below, which is standard modern Romanian)
_DIACRITIC_MAP = str.maketrans(
    'şŞţŢ',   # cedilla variants
    'șȘțȚ',   # comma-below variants
)

def normalise_diacritics(text: str) -> str:
    text = text.translate(_DIACRITIC_MAP)
    # Also decompose + recompose to handle Unicode edge cases
    text = unicodedata.normalize('NFC', text)
    return text


def clean_text(text: str) -> str:
    text = remove_html(text)
    text = normalise_diacritics(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalise_label(raw: str) -> str:
    key = str(raw).lower().strip()
    return LABEL_MAP_NORM.get(key, 'unknown')


print('Cleaning utilities defined.')

## 2. Load EuroVerdict (Romanian subset)

In [ ]:
from datasets import load_dataset

# EuroVerdict is hosted on HuggingFace — load the Romanian language split
try:
    euro_ds = load_dataset('Cartinoe5930/EuroVerdict', split='train')
    euro_df = euro_ds.to_pandas()
    print(f'EuroVerdict raw shape: {euro_df.shape}')
    print(euro_df.columns.tolist())
except Exception as e:
    print(f'Could not load EuroVerdict from HuggingFace: {e}')
    print('Attempting to load from local file data/raw/euroverdict_ro.csv …')
    euro_df = pd.read_csv(RAW_DIR / 'euroverdict_ro.csv')

In [ ]:
# Filter Romanian rows if a language column exists
lang_col = next((c for c in euro_df.columns if 'lang' in c.lower()), None)
if lang_col:
    euro_df = euro_df[euro_df[lang_col].str.lower().str.startswith('ro')].copy()
    print(f'After Romanian filter: {euro_df.shape}')

# Map to standard schema — adjust column names to actual dataset columns
EURO_COL_MAP = {
    'claim':       'claim_text',
    'article':     'evidence_text',
    'label':       'veracity_label',
    'verdict':     'justification',
}

rename_dict = {k: v for k, v in EURO_COL_MAP.items() if k in euro_df.columns}
euro_df = euro_df.rename(columns=rename_dict)

for col in ['claim_text', 'evidence_text', 'veracity_label', 'justification']:
    if col not in euro_df.columns:
        euro_df[col] = ''

euro_df['source'] = 'euroverdict'
euro_df['claim_id'] = [f'ev_{uuid.uuid4().hex[:8]}' for _ in range(len(euro_df))]

print(euro_df[['claim_id', 'claim_text', 'veracity_label']].head(3))

## 3. Load PolyTruth (Romanian subset)

In [ ]:
try:
    poly_ds = load_dataset('Cartinoe5930/PolyTruth', split='train')
    poly_df = poly_ds.to_pandas()
    print(f'PolyTruth raw shape: {poly_df.shape}')
    print(poly_df.columns.tolist())
except Exception as e:
    print(f'Could not load PolyTruth from HuggingFace: {e}')
    poly_df = pd.read_csv(RAW_DIR / 'polytruth_ro.csv')

lang_col = next((c for c in poly_df.columns if 'lang' in c.lower()), None)
if lang_col:
    poly_df = poly_df[poly_df[lang_col].str.lower().str.startswith('ro')].copy()
    print(f'After Romanian filter: {poly_df.shape}')

In [ ]:
POLY_COL_MAP = {
    'statement':    'claim_text',
    'correction':   'justification',
    'label':        'veracity_label',
    'context':      'evidence_text',
}
rename_dict = {k: v for k, v in POLY_COL_MAP.items() if k in poly_df.columns}
poly_df = poly_df.rename(columns=rename_dict)

for col in ['claim_text', 'evidence_text', 'veracity_label', 'justification']:
    if col not in poly_df.columns:
        poly_df[col] = ''

poly_df['source'] = 'polytruth'
poly_df['claim_id'] = [f'pt_{uuid.uuid4().hex[:8]}' for _ in range(len(poly_df))]

print(poly_df[['claim_id', 'claim_text', 'veracity_label']].head(3))

## 4. Load dezinformare-ro (rares127/dezinformare-ro)

In [ ]:
try:
    dezinfo_ds = load_dataset('rares127/dezinformare-ro', split='train')
    dezinfo_df = dezinfo_ds.to_pandas()
    print(f'dezinformare-ro raw shape: {dezinfo_df.shape}')
    print(dezinfo_df.columns.tolist())
except Exception as e:
    print(f'Could not load dezinformare-ro: {e}')
    dezinfo_df = pd.read_csv(RAW_DIR / 'dezinformare_ro.csv')

In [ ]:
DEZINFO_COL_MAP = {
    'claim':        'claim_text',
    'text':         'claim_text',
    'label':        'veracity_label',
    'evidence':     'evidence_text',
    'justification':'justification',
    'explanation':  'justification',
}
rename_dict = {k: v for k, v in DEZINFO_COL_MAP.items() if k in dezinfo_df.columns}
dezinfo_df = dezinfo_df.rename(columns=rename_dict)

for col in ['claim_text', 'evidence_text', 'veracity_label', 'justification']:
    if col not in dezinfo_df.columns:
        dezinfo_df[col] = ''

dezinfo_df['source'] = 'dezinformare-ro'
dezinfo_df['claim_id'] = [f'dz_{uuid.uuid4().hex[:8]}' for _ in range(len(dezinfo_df))]

print(dezinfo_df[['claim_id', 'claim_text', 'veracity_label']].head(3))

## 5. Optional: Load stopfals.md scraped data

In [ ]:
import json

stopfals_path = RAW_DIR / 'stopfals.jsonl'
if stopfals_path.exists():
    records = [json.loads(l) for l in stopfals_path.read_text(encoding='utf-8').splitlines() if l]
    sf_df = pd.DataFrame(records)
    sf_df = sf_df.rename(columns={'verdict': 'veracity_label'})
    for col in ['claim_text', 'evidence_text', 'veracity_label', 'justification']:
        if col not in sf_df.columns:
            sf_df[col] = ''
    sf_df['source'] = 'stopfals'
    sf_df['claim_id'] = [f'sf_{uuid.uuid4().hex[:8]}' for _ in range(len(sf_df))]
    print(f'stopfals.md records: {len(sf_df)}')
else:
    print('No stopfals.jsonl found — run src/scraper.py if more data is needed.')
    sf_df = pd.DataFrame(columns=['claim_id', 'claim_text', 'evidence_text', 'veracity_label', 'justification', 'source'])

## 6. Merge, Clean & Standardise

In [ ]:
KEEP_COLS = ['claim_id', 'claim_text', 'evidence_text', 'veracity_label', 'justification', 'source']

frames = []
for df in [euro_df, poly_df, dezinfo_df, sf_df]:
    sub = df[[c for c in KEEP_COLS if c in df.columns]].copy()
    for c in KEEP_COLS:
        if c not in sub.columns:
            sub[c] = ''
    frames.append(sub)

combined = pd.concat(frames, ignore_index=True)
print(f'Combined shape before cleaning: {combined.shape}')

In [ ]:
# Apply text cleaning
for col in ['claim_text', 'evidence_text', 'justification']:
    combined[col] = combined[col].fillna('').apply(clean_text)

# Normalise labels
combined['veracity_label'] = combined['veracity_label'].apply(normalise_label)

# Drop rows with unknown labels or empty claim text
before = len(combined)
combined = combined[combined['veracity_label'] != 'unknown']
combined = combined[combined['claim_text'].str.len() > 10]
combined = combined.drop_duplicates(subset='claim_text')
print(f'Dropped {before - len(combined)} rows (unknown labels / empty claims / duplicates)')
print(f'Final dataset shape: {combined.shape}')
print(combined['veracity_label'].value_counts())

## 7. Stratified 80/10/10 Split

No article text overlap across splits to prevent data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

combined = combined.reset_index(drop=True)

train_df, temp_df = train_test_split(
    combined,
    test_size=0.20,
    stratify=combined['veracity_label'],
    random_state=42,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['veracity_label'],
    random_state=42,
)

print(f'Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}')
print('Train label dist:')
print(train_df['veracity_label'].value_counts())

# Verify no evidence_text overlap between train and test
train_evidence = set(train_df['evidence_text'].dropna())
test_overlap = test_df['evidence_text'].isin(train_evidence).sum()
print(f'Evidence text overlap (train ∩ test): {test_overlap} rows')

In [ ]:
train_df.to_csv(PROCESSED_DIR / 'train.csv', index=False, encoding='utf-8')
val_df.to_csv(PROCESSED_DIR / 'val.csv', index=False, encoding='utf-8')
test_df.to_csv(PROCESSED_DIR / 'test.csv', index=False, encoding='utf-8')

print('Saved:')
print(f'  {PROCESSED_DIR}/train.csv ({len(train_df)} rows)')
print(f'  {PROCESSED_DIR}/val.csv   ({len(val_df)} rows)')
print(f'  {PROCESSED_DIR}/test.csv  ({len(test_df)} rows)')